# 07 · Análise preditiva (classificação)

> **Atenção:** execute todas as células (Kernel → Restart & Run All) e salve o notebook **com as saídas** antes de enviar ao GitHub. O curso exige células de código e de saída visíveis.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

data = pd.read_csv('../data/dataset_part_2.csv')
X = pd.read_csv('../data/dataset_part_3.csv')
Y = data['Class'].to_numpy()
X = preprocessing.StandardScaler().fit(X).transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
Y_test.shape

In [ ]:
def plot_confusion_matrix(y, y_predict):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot=True, ax=ax, fmt='g')
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
    ax.xaxis.set_ticklabels(['não pousou', 'pousou']); ax.yaxis.set_ticklabels(['não pousou', 'pousou'])
    plt.show()

In [ ]:
models = {
 'Regressão Logística': (LogisticRegression(), {'C':[0.01,0.1,1], 'penalty':['l2'], 'solver':['lbfgs']}),
 'SVM': (SVC(), {'kernel':('linear','rbf','poly','sigmoid'), 'C':np.logspace(-3,3,5), 'gamma':np.logspace(-3,3,5)}),
 'Árvore de Decisão': (DecisionTreeClassifier(), {'criterion':['gini','entropy'], 'splitter':['best','random'],
     'max_depth':[2*n for n in range(1,10)], 'max_features':['sqrt','log2'], 'min_samples_leaf':[1,2,4], 'min_samples_split':[2,5,10]}),
 'KNN': (KNeighborsClassifier(), {'n_neighbors':list(range(1,11)), 'algorithm':['auto','ball_tree','kd_tree','brute'], 'p':[1,2]}),
}
results, fitted = {}, {}
for name, (model, params) in models.items():
    g = GridSearchCV(model, params, cv=10).fit(X_train, Y_train)
    fitted[name] = g
    results[name] = {'melhor score (validação cruzada)': g.best_score_, 'acurácia no teste': g.score(X_test, Y_test)}
pd.DataFrame(results).T

In [ ]:
pd.DataFrame(results).T['acurácia no teste'].plot(kind='bar', ylim=(0,1))
plt.ylabel('Acurácia no teste'); plt.show()

In [ ]:
best = max(results, key=lambda k: results[k]['melhor score (validação cruzada)'])
print('Melhor modelo (validação cruzada):', best)
plot_confusion_matrix(Y_test, fitted[best].predict(X_test))